# Kiểm chứng star schema theo PS1–PS5 và hợp đồng mở rộng

Tài liệu đi cùng: [thiết kế chuẩn](../../docs/star_schema.md), [truy vết PS](../../docs/star_schema_tu_ps.md).
Nguồn nghiệp vụ: deck `docs/presentations/revenue_performance_problem_statement_updated.pptx`.

Notebook độc lập, **không build dbt, không ghi vào warehouse**. CSV được đọc trực tiếp; database hiện có
được attach read-only; SQL kiểm tra chạy trong DuckDB bộ nhớ. Mọi assertion phải PASS mới chấp nhận.
Không xác nhận deployment Snowflake, Kafka, SCD2 hay tính nhân quả. Không đổi file nguồn / loại outlier.

Chạy từ repo root hoặc thư mục notebook bằng kernel môi trường `.venv`. Cần `data/*.csv` và
`warehouse/dbt.duckdb` có schema marts/reporting; thiếu đầu vào phải dừng, không tự rebuild.


In [1]:
from pathlib import Path
from decimal import Decimal
import hashlib
import json
import platform
import datetime as dt
import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "retail_dbt").is_dir() and (p / "data").is_dir()), None)
assert ROOT is not None, "Không tìm được repo root và data/"
DB = ROOT / "warehouse/dbt.duckdb"
assert DB.is_file(), "Thiếu warehouse/dbt.duckdb; notebook không tự dựng lại kho"
checks = []
def check(name, condition):
    passed = bool(condition)
    checks.append({"check": name, "passed": passed})
    assert passed, name

def frame(sql):
    # fetchall giữ Decimal thay vì tự đổi money sang float.
    result = con.execute(sql)
    return pd.DataFrame(result.fetchall(), columns=[d[0] for d in result.description])

def zero(name, sql):
    check(name, con.execute(sql).fetchone()[0] == 0)

def close(name, actual, expected, atol=1e-11):
    check(name, np.allclose(np.asarray(actual, dtype=float), np.asarray(expected, dtype=float),
                            rtol=0, atol=atol, equal_nan=True))

def cents(series):
    return series.map(lambda v: None if pd.isna(v) else int(Decimal(str(v)) * 100))

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

source_names = ['customers','geography','inventory','orders','order_items','payments','products',
                'promotions','returns','reviews','sales','sample_submission','shipments','web_traffic']
source_files = [ROOT / 'data' / (name + '.csv') for name in sorted(source_names)]
assert all(p.is_file() for p in source_files), 'Thiếu một trong 14 nguồn theo hợp đồng'
print('Không nạp submission.csv hoặc file xuất khác làm nguồn warehouse.')
manifest_files = source_files + [DB, ROOT / "docs/presentations/revenue_performance_problem_statement_updated.pptx"]
manifest = pd.DataFrame([{"path": str(p.relative_to(ROOT)), "bytes": p.stat().st_size,
                          "sha256": sha256(p)} for p in manifest_files])
print("Kiểm tra lúc:", dt.datetime.now(dt.timezone.utc).isoformat())
print("Python", platform.python_version(), "| DuckDB", duckdb.__version__, "| pandas", pd.__version__)
display(manifest)
con = duckdb.connect(":memory:")
db_path = DB.as_posix().replace("'", "''")
con.execute(f"ATTACH '{db_path}' AS dbt (READ_ONLY)")
for path in source_files:
    source_path = path.as_posix().replace("'", "''")
    con.execute(f"CREATE TABLE raw_{path.stem} AS SELECT * FROM read_csv('{source_path}', all_varchar=true)")
check("14 file CSV trong snapshot", len(source_files) == 14)


Không nạp submission.csv hoặc file xuất khác làm nguồn warehouse.


Kiểm tra lúc: 2026-09-27T00:16:19.937112+00:00
Python 3.13.5 | DuckDB 1.5.5 | pandas 3.0.5


,path,bytes,sha256
0,data\customers.csv,7079679,5e58abf49e99d52bf8ddda44601a30e78667760908b862...
1,data\geography.csv,1402281,f5d150bdfdfca7ac0c38118b1f5f8ea1b3f5de4cd23fa7...
2,data\inventory.csv,5668678,1bbcdd89569a5df4b938ac0577f0657030c41c3a18956f...
3,data\order_items.csv,23942735,a8b2adfe54eec0dda79ced28e5987d6846b7f31cb60523...
4,data\orders.csv,45963232,f4b3029f386f5f5a6baabd78aad0140e5a101ea6975f4c...
5,data\payments.csv,18383288,aacef7716fe1c1442bc46e8885019cf1d67e12655c2e0f...
6,data\products.csv,195223,890946808d8237dad0d39153efd427f5399a5e6eaa2302...
7,data\promotions.csv,4444,158ee5843ee41a28b5f5748ab2c73bd712524da6c00930...
8,data\returns.csv,2281456,501717bc76cf237c0dbbb68f586351870047198042c423...
9,data\reviews.csv,6791414,5230859408955d611bc430df5d788eb66fedd56f3be9f6...


## 1. Grain, khóa và cardinality

Số thứ tự nguồn được lấy trực tiếp từ thứ tự CSV, không đọc intermediate của dbt. Giữ mọi dòng, kể cả
cặp order/product lặp. Kiểm uniqueness/NULL của toàn bộ 14 bảng marts; kiểm FK và bản join dùng cho PS.


In [2]:
# pandas giữ thứ tự file, source_row được thêm trước khi sắp xếp.
source_items = pd.read_csv(ROOT / "data/order_items.csv", dtype=str, keep_default_na=False)
source_items["source_row"] = np.arange(1, len(source_items) + 1)
source_items["line_number"] = source_items.groupby("order_id", sort=False).cumcount() + 1
con.register("source_items", source_items)
con.execute("""
CREATE TABLE expected_detail AS
SELECT CAST(i.order_id AS INTEGER) AS order_id, i.line_number,
       CAST(i.product_id AS INTEGER) AS product_id, CAST(o.customer_id AS INTEGER) AS customer_id,
       CAST(o.order_date AS DATE) AS order_date, o.order_status,
       p.category, c.acquisition_channel, g.region,
       CAST(i.quantity AS INTEGER) AS quantity,
       CAST(i.unit_price AS DECIMAL(14,2)) AS unit_price,
       CAST(i.discount_amount AS DECIMAL(16,2)) AS discount_amount,
       CAST(i.quantity AS INTEGER) * CAST(i.unit_price AS DECIMAL(14,2)) AS gross_amount,
       CAST(i.quantity AS INTEGER) * CAST(i.unit_price AS DECIMAL(14,2))
           - CAST(i.discount_amount AS DECIMAL(16,2)) AS net_amount
FROM source_items i
JOIN raw_orders o ON o.order_id = i.order_id
JOIN raw_products p ON p.product_id = i.product_id
JOIN raw_customers c ON c.customer_id = o.customer_id
JOIN raw_geography g ON g.zip = c.zip
""")
check("Nguồn join không mất/nhân dòng", con.sql("SELECT count(*) FROM expected_detail").fetchone()[0] == len(source_items))
duplicate_pairs = source_items.groupby(["order_id", "product_id"]).size()
print("Cặp (order_id, product_id) lặp:", int((duplicate_pairs > 1).sum()))
check("Phản ví dụ order/product không phải PK", (duplicate_pairs > 1).any())
check("Khóa tái dựng order/line unique", not source_items.duplicated(["order_id", "line_number"]).any())

keys = {
    "dim_date": ["date_sk"], "dim_product": ["product_sk"], "dim_customer": ["customer_sk"],
    "dim_geography": ["geography_sk"], "dim_order_junk": ["order_junk_sk"],
    "dim_promotion": ["promotion_sk"], "fact_order_item": ["order_item_sk"],
    "fact_order": ["order_id"], "fact_return": ["return_id"], "fact_review": ["review_id"],
    "fact_daily_sales": ["date_sk"], "fact_web_traffic": ["date_sk"],
    "fact_inventory_snapshot": ["date_sk", "product_sk"],
    "bridge_item_promo": ["order_item_sk", "promotion_sk"],
}
counts = []
for table, columns in keys.items():
    cols = ", ".join(columns)
    nulls = " OR ".join(f"{c} IS NULL" for c in columns)
    zero(f"{table}: PK không NULL", f"SELECT count(*) FROM dbt.marts.{table} WHERE {nulls}")
    zero(f"{table}: PK unique", f"SELECT count(*) FROM (SELECT {cols} FROM dbt.marts.{table} GROUP BY {cols} HAVING count(*) > 1)")
    counts.append({"table": table, "rows": con.sql(f"SELECT count(*) FROM dbt.marts.{table}").fetchone()[0]})
zero("fact_order_item: order/line unique", """
SELECT count(*) FROM (SELECT order_id,line_number FROM dbt.marts.fact_order_item
GROUP BY order_id,line_number HAVING count(*) <> 1)""")
fk_contract = [
 ("fact_order_item","date_sk","dim_date","date_sk"), ("fact_order_item","product_sk","dim_product","product_sk"),
 ("fact_order_item","customer_sk","dim_customer","customer_sk"), ("fact_order_item","order_junk_sk","dim_order_junk","order_junk_sk"),
 ("dim_customer","geography_sk","dim_geography","geography_sk"),
 ("fact_order","customer_sk","dim_customer","customer_sk"), ("fact_order","order_junk_sk","dim_order_junk","order_junk_sk"),
 ("fact_order","order_date_sk","dim_date","date_sk"), ("fact_order","ship_date_sk","dim_date","date_sk"),
 ("fact_order","delivery_date_sk","dim_date","date_sk"),
 ("fact_return","order_item_sk","fact_order_item","order_item_sk"), ("fact_review","order_item_sk","fact_order_item","order_item_sk"),
 ("fact_return","date_sk","dim_date","date_sk"), ("fact_review","date_sk","dim_date","date_sk"),
 ("fact_inventory_snapshot","date_sk","dim_date","date_sk"), ("fact_inventory_snapshot","product_sk","dim_product","product_sk"),
 ("fact_daily_sales","date_sk","dim_date","date_sk"), ("fact_web_traffic","date_sk","dim_date","date_sk"),
 ("bridge_item_promo","order_item_sk","fact_order_item","order_item_sk"), ("bridge_item_promo","promotion_sk","dim_promotion","promotion_sk"),
]
for child, fk, parent, pk in fk_contract:
    zero(f"FK {child}.{fk} -> {parent}.{pk}", f"""SELECT count(*) FROM dbt.marts.{child} c
    LEFT JOIN dbt.marts.{parent} p ON c.{fk}=p.{pk} WHERE c.{fk} IS NOT NULL AND p.{pk} IS NULL""")
display(pd.DataFrame(counts))


Cặp (order_id, product_id) lặp: 16


,table,rows
0,dim_date,4566
1,dim_product,2412
2,dim_customer,121930
3,dim_geography,39948
4,dim_order_junk,540
5,dim_promotion,50
6,fact_order_item,714669
7,fact_order,646945
8,fact_return,39939
9,fact_review,113551


## 2. Đối chiếu lõi với nguồn độc lập

So từng dòng bằng phép hiệu hai chiều, bao gồm khóa ghép, ID nghiệp vụ, ngày, trạng thái,
category/region/acquisition_channel và tiền kiểu DECIMAL. Không lấy bảng aggregate do cùng pipeline tạo ra
làm đối chứng duy nhất. Geography đi qua customer, không giả định FK geography nằm ở fact.


In [3]:
con.execute("""
CREATE VIEW actual_detail AS
SELECT f.order_id, f.line_number, p.product_id, c.customer_id,
       d.full_date AS order_date, j.order_status, p.category, c.acquisition_channel, g.region,
       f.quantity, f.unit_price, f.discount_amount, f.gross_amount, f.net_amount
FROM dbt.marts.fact_order_item f
JOIN dbt.marts.dim_date d USING(date_sk)
JOIN dbt.marts.dim_order_junk j USING(order_junk_sk)
JOIN dbt.marts.dim_product p USING(product_sk)
JOIN dbt.marts.dim_customer c USING(customer_sk)
JOIN dbt.marts.dim_geography g USING(geography_sk)
""")
for left, right in [("expected_detail","actual_detail"), ("actual_detail","expected_detail")]:
    zero(f"Đối chiếu dòng: {left} trừ {right}", f"SELECT count(*) FROM (SELECT * FROM {left} EXCEPT ALL SELECT * FROM {right})")
row_count = con.sql("SELECT count(*) FROM dbt.marts.fact_order_item").fetchone()[0]
check("Join 5 dimension giữ nguyên grain", con.sql("SELECT count(*) FROM actual_detail").fetchone()[0] == row_count)
check("714669 dòng đúng snapshot", row_count == 714669)
zero("Reporting detail không nhân order_item_sk", """
SELECT count(*) FROM (SELECT order_item_sk FROM dbt.reporting.int_reporting_order_items
GROUP BY order_item_sk HAVING count(*) <> 1)""")
check("Reporting detail đủ dòng", con.sql("SELECT count(*) FROM dbt.reporting.int_reporting_order_items").fetchone()[0] == row_count)
zero("Reporting detail khớp thuộc tính và measure", """
SELECT count(*) FROM dbt.reporting.int_reporting_order_items r
JOIN dbt.marts.fact_order_item f USING(order_item_sk)
JOIN actual_detail a ON a.order_id=f.order_id AND a.line_number=f.line_number
WHERE r.order_id IS DISTINCT FROM a.order_id OR r.customer_sk IS DISTINCT FROM f.customer_sk
OR r.order_date IS DISTINCT FROM a.order_date OR r.order_status IS DISTINCT FROM a.order_status
OR r.is_delivered IS DISTINCT FROM (a.order_status='delivered')
OR r.category IS DISTINCT FROM a.category OR r.region IS DISTINCT FROM a.region
OR r.acquisition_channel IS DISTINCT FROM a.acquisition_channel
OR r.quantity IS DISTINCT FROM a.quantity OR r.gross_amount IS DISTINCT FROM a.gross_amount
OR r.discount_amount IS DISTINCT FROM a.discount_amount OR r.net_amount IS DISTINCT FROM a.net_amount
""")
display(frame("SELECT * FROM actual_detail ORDER BY order_id,line_number LIMIT 3"))


,order_id,line_number,product_id,customer_id,order_date,order_status,category,acquisition_channel,region,quantity,unit_price,discount_amount,gross_amount,net_amount
0,1,1,2400,58578,2012-07-04,delivered,GenZ,social_media,East,7,1138.22,0.00,7967.54,7967.54
1,2,1,609,58621,2012-07-04,returned,Streetwear,social_media,East,7,10166.25,0.00,71163.75,71163.75
2,3,1,396,58811,2012-07-04,delivered,Streetwear,direct,East,3,11220.33,0.00,33660.99,33660.99


## 3. R/G, lịch sử đầy đủ và đối chứng sales/payments

Tiền đối soát bằng số nguyên phần trăm đơn vị tiền (cents), không dùng sai số tương đối trên tổng hàng tỷ.
R chỉ delivered; Q/N/C cùng tập lọc. Returned bị loại nguyên đơn, không trừ refund lần hai.


In [4]:
detail = frame("""
SELECT *, CAST(gross_amount * 100 AS BIGINT) AS g_cents,
          CAST(net_amount * 100 AS BIGINT) AS net_cents,
          CAST(discount_amount * 100 AS BIGINT) AS discount_cents
FROM expected_detail
""")
detail["order_date"] = pd.to_datetime(detail["order_date"])
detail["year"] = detail.order_date.dt.year
detail["month_start_date"] = detail.order_date.dt.to_period("M").dt.to_timestamp()
detail["delivered"] = detail.order_status.eq("delivered")
detail["r_cents"] = detail.net_cents.where(detail.delivered, 0)
detail["delivered_discount_cents"] = detail.discount_cents.where(detail.delivered, 0)
detail["cancelled_gross_cents"] = detail.g_cents.where(detail.order_status.eq("cancelled"), 0)
detail["returned_gross_cents"] = detail.g_cents.where(detail.order_status.eq("returned"), 0)
detail["undelivered_gross_cents"] = detail.g_cents.where(detail.order_status.isin(["created","paid","shipped"]), 0)
detail["r_day26_plus_cents"] = detail.r_cents.where(detail.order_date.dt.day.ge(26), 0)
check("Không có trạng thái ngoài hợp đồng", set(detail.order_status) == {"created","paid","shipped","delivered","cancelled","returned"})
check("G-R theo trạng thái và chiết khấu", (
    detail.g_cents-detail.r_cents == detail.cancelled_gross_cents+detail.returned_gross_cents
    +detail.undelivered_gross_cents+detail.delivered_discount_cents).all())
zero("Daily G bằng sales.csv (DECIMAL)", """
WITH a AS (SELECT order_date, SUM(gross_amount) AS g FROM expected_detail GROUP BY 1),
b AS (SELECT CAST("Date" AS DATE) AS order_date,CAST("Revenue" AS DECIMAL(18,2)) AS g FROM raw_sales)
SELECT count(*) FROM a FULL JOIN b USING(order_date) WHERE a.g IS DISTINCT FROM b.g""")
zero("R từng đơn bằng payments.csv của đơn delivered", """
WITH a AS (SELECT order_id,SUM(net_amount) AS r FROM expected_detail WHERE order_status='delivered' GROUP BY 1),
b AS (SELECT CAST(p.order_id AS INTEGER) AS order_id, CAST(p.payment_value AS DECIMAL(18,2)) AS r
      FROM raw_payments p JOIN raw_orders o USING(order_id) WHERE o.order_status='delivered')
SELECT count(*) FROM a FULL JOIN b USING(order_id) WHERE a.r IS DISTINCT FROM b.r""")
check("G toàn lịch sử", detail.g_cents.sum() == 1643047658553)
check("R toàn lịch sử", detail.r_cents.sum() == 1251817595720)
check("R 2013-2022", detail.loc[detail.year.between(2013,2022),"r_cents"].sum() == 1192640346890)
zero("fact_daily_sales actual là G, không phải R", """
WITH a AS (SELECT order_date,SUM(gross_amount) AS g FROM expected_detail GROUP BY 1),
b AS (SELECT d.full_date AS order_date,f.revenue AS g FROM dbt.marts.fact_daily_sales f
JOIN dbt.marts.dim_date d USING(date_sk) WHERE is_actual)
SELECT count(*) FROM a FULL JOIN b USING(order_date) WHERE a.g IS DISTINCT FROM b.g""")
zero("Vùng non-actual khớp mẫu, không suy diễn dự báo", """
WITH a AS (SELECT d.full_date AS day,f.revenue,f.cogs FROM dbt.marts.fact_daily_sales f
JOIN dbt.marts.dim_date d USING(date_sk) WHERE NOT is_actual),
b AS (SELECT CAST("Date" AS DATE) AS day,CAST("Revenue" AS DECIMAL(18,2)) AS revenue,
CAST("COGS" AS DECIMAL(18,2)) AS cogs FROM raw_sample_submission)
SELECT count(*) FROM a FULL JOIN b USING(day)
WHERE a.revenue IS DISTINCT FROM b.revenue OR a.cogs IS DISTINCT FROM b.cogs""")
display(detail.groupby("order_status").agg(lines=("order_id","size"),orders=("order_id","nunique"),
    gross_cents=("g_cents","sum"),revenue_cents=("r_cents","sum")))
print("G =", Decimal(int(detail.g_cents.sum()))/100, "| R =", Decimal(int(detail.r_cents.sum()))/100)


,lines,orders,gross_cents,revenue_cents
order_status,,,,
cancelled,65673,59462,151589100619,0
created,7994,7275,19065233076,0
delivered,570887,516716,1311709136041,1251817595720
paid,14987,13577,34592285499,0
returned,40034,36142,90754303193,0
shipped,15094,13773,35337600125,0


G = 16430476585.53 | R = 12518175957.2


## 4. Reporting tháng/năm: tính độc lập từ detail nguồn

N và C được đếm lại cho từng grain. Các tỷ lệ dùng tổng tử/tổng mẫu; kiểm cả mặt nạ NULL.
Không lấy tổng C tháng để làm C năm.


In [5]:
money_columns = ["g","r","cancelled_gross","returned_gross","undelivered_gross","delivered_discount","r_day26_plus"]
def aggregate(key):
    grouped = detail.groupby(key)
    result = grouped[[c+"_cents" for c in money_columns]].sum()
    delivered = detail[detail.delivered].groupby(key).agg(n=("order_id","nunique"),q=("quantity","sum"),c=("customer_id","nunique"))
    result = result.join(delivered).fillna(0).sort_index()
    for metric in money_columns:
        result[metric] = result[metric+"_cents"] / 100
    result["u"] = result.q/result.n.replace(0,np.nan)
    result["p"] = result.r/result.q.replace(0,np.nan)
    result["f"] = result.n/result.c.replace(0,np.nan)
    result["aov"] = result.r/result.n.replace(0,np.nan)
    result["capture_rate"] = result.r/result.g.replace(0,np.nan)
    result["cancelled_rate"] = result.cancelled_gross/result.g.replace(0,np.nan)
    result["discount_rate"] = result.delivered_discount/(result.r+result.delivered_discount).replace(0,np.nan)
    return result

expected_month = aggregate("month_start_date")
expected_year = aggregate("year")
actual_month = frame("SELECT * FROM dbt.reporting.rpt_revenue_monthly ORDER BY month_start_date")
actual_month["month_start_date"] = pd.to_datetime(actual_month.month_start_date)
actual_month = actual_month.set_index("month_start_date")
actual_year = frame("SELECT * FROM dbt.reporting.rpt_revenue_yearly ORDER BY year").set_index("year")
for grain, actual, expected in [("tháng",actual_month,expected_month),("năm",actual_year,expected_year)]:
    check(f"{grain}: index và không trùng grain", actual.index.equals(expected.index) and actual.index.is_unique)
    for metric in money_columns:
        if metric in actual:
            check(f"{grain}: {metric} chính xác cents", np.array_equal(cents(actual[metric]),expected[metric+"_cents"]))
    for metric in ["n","q","c"]:
        check(f"{grain}: {metric} đếm từ nguồn", np.array_equal(actual[metric],expected[metric]))
    for metric in ["u","p","f","aov","capture_rate","cancelled_rate","discount_rate"]:
        close(f"{grain}: {metric}", actual[metric],expected[metric])
    years = actual.index.year if grain == "tháng" else actual.index
    check(f"{grain}: is_analysis_period", np.array_equal(actual.is_analysis_period, (years>=2013)&(years<=2022)))
    close(f"{grain}: R=N*U*P", expected.n*expected.u*expected.p,expected.r,atol=1e-5)
    close(f"{grain}: N=C*F", expected.c*expected.f,expected.n,atol=1e-8)
display(actual_year[["g","r","n","q","c","u","p","f"]])


,g,r,n,q,c,u,p,f
year,,,,,,,,
2012,741497748.02,591772488.30,25663,133167,18759,5.189066,4443.837349,1.368037
2013,1657169417.27,1259163189.54,61588,314571,34806,5.107667,4002.794884,1.769465
2014,1871845882.80,1433063281.22,64688,328810,35673,5.083014,4358.332415,1.813360
2015,1889933826.92,1436004017.22,66163,333203,36050,5.036093,4309.697143,1.835312
2016,2104640677.54,1619505655.64,66067,329162,36242,4.982245,4920.086935,1.822940
2017,1911164325.28,1461976298.95,61144,302016,34957,4.939422,4840.724660,1.749120
2018,1850122456.08,1419275129.27,55740,270404,33307,4.851166,5248.720911,1.673522
2019,1136801441.51,864329801.94,33259,161837,23401,4.865961,5340.742858,1.421264
2020,1054512158.79,806414699.14,27975,133785,20839,4.782306,6027.691439,1.342435


## 5. Calendar, mùa vụ và PS2

Kiểm tháng liên tục, ngày nhuận, mặt nạ năm thiếu, rolling/YoY và August parity.
Không tự chọn các giai đoạn CAGR hoặc kết luận causal từ lịch.


In [6]:
date_dim = frame("SELECT full_date, month_start_date, days_in_month, date_sk FROM dbt.marts.dim_date ORDER BY full_date")
date_dim["full_date"] = pd.to_datetime(date_dim.full_date)
check("dim_date đầy đủ từng ngày", date_dim.full_date.tolist() == pd.date_range(date_dim.full_date.min(),date_dim.full_date.max()).tolist())
check("date_sk đúng YYYYMMDD", np.array_equal(date_dim.date_sk,date_dim.full_date.dt.strftime("%Y%m%d").astype(int)))
check("days_in_month đúng lịch có năm nhuận", np.array_equal(date_dim.days_in_month,date_dim.full_date.dt.days_in_month))
check("month_start_date đúng lịch", np.array_equal(pd.to_datetime(date_dim.month_start_date),date_dim.full_date.dt.to_period("M").dt.to_timestamp()))
check("126 tháng liên tục", actual_month.index.equals(pd.date_range("2012-07-01","2022-12-01",freq="MS")))
check("is_odd_year", np.array_equal(actual_month.is_odd_year,actual_month.index.year % 2 == 1))
check("month/year tháng đúng key", np.array_equal(actual_month.month,actual_month.index.month) and np.array_equal(actual_month.year,actual_month.index.year))
check("Số ngày tháng reporting", np.array_equal(actual_month.days_in_month,actual_month.index.days_in_month))
expected_month["month_index"] = expected_month.r / expected_month.groupby(expected_month.index.year).r.transform("mean")
expected_month.loc[expected_month.index.year < 2013,"month_index"] = np.nan
expected_month["r_12m"] = expected_month.r.rolling(12).sum()
expected_month.loc[expected_month.index < "2013-07-01","r_12m"] = np.nan
expected_month["r_same_month_prior_year"] = expected_month.r.shift(12)
expected_month.loc[expected_month.index < "2013-08-01","r_same_month_prior_year"] = np.nan
expected_month["yoy_rate"] = expected_month.r/expected_month.r_same_month_prior_year-1
expected_month["eom_share"] = expected_month.r_day26_plus/expected_month.r
expected_month["eom_expected_share"] = (expected_month.index.days_in_month-25)/expected_month.index.days_in_month
expected_month["eom_excess"] = expected_month.eom_share-expected_month.eom_expected_share
for metric in ["r_12m","r_same_month_prior_year"]:
    actual_values = actual_month[metric]
    check(f"{metric}: mặt nạ NULL", actual_values.isna().equals(expected_month[metric].isna()))
    mask = actual_values.notna()
    check(f"{metric}: exact cents", np.array_equal(cents(actual_values[mask]),(expected_month.loc[mask,metric]*100).round().astype("int64")))
for metric in ["month_index","yoy_rate","eom_share","eom_expected_share","eom_excess"]:
    close("Calendar "+metric, actual_month[metric],expected_month[metric])
expected_year["yoy_rate"] = expected_year.r/expected_year.r.shift()-1
expected_year.loc[expected_year.index<2014,"yoy_rate"] = np.nan
close("YoY năm chỉ từ 2014",actual_year.yoy_rate,expected_year.yoy_rate)
august = expected_month.loc[(expected_month.index.month==8)&(expected_month.index.year>=2013),"month_index"]
odd, even = august[august.index.year%2==1], august[august.index.year%2==0]
parity = frame("SELECT * FROM dbt.reporting.rpt_august_parity")
check("August parity grain một dòng", len(parity)==1)
check("August đủ 5 năm lẻ, 5 năm chẵn", parity.n_odd_years.iloc[0]==len(odd)==5 and parity.n_even_years.iloc[0]==len(even)==5)
close("August index năm lẻ",parity.august_index_odd,[odd.mean()])
close("August index năm chẵn",parity.august_index_even,[even.mean()])
close("August odd/even",parity.august_odd_vs_even,[odd.mean()/even.mean()-1])
season = expected_month.groupby(expected_month.index.year).month_index.agg(["min","max"])
close("Season peak/trough",actual_year.season_peak_trough_ratio,season["max"]/season["min"])
display(actual_month.loc["2020-02-01":"2020-03-01",["days_in_month","eom_share","eom_expected_share","month_index"]])
display(parity)


,days_in_month,eom_share,eom_expected_share,month_index
month_start_date,,,,
2020-02-01,29,0.134961,0.137931,0.750186
2020-03-01,31,0.299366,0.193548,1.265030


,n_odd_years,n_even_years,august_index_odd,august_index_even,august_odd_vs_even
0,5,5,0.831694,1.334138,-0.376606


## 6. PS4/PS5: phân rã số học và ba lát cắt độc lập

Tính từng thành phần đúng thứ tự N → U → P và C → F, không chỉ kiểm tổng.
Segment bao gồm lưới nhóm × năm, cả năm doanh thu nhóm bằng 0.


In [7]:
e = expected_year
previous = e.shift()
mask = e.index >= 2014
formulas = {
    "contrib_n": (e.n-previous.n)*previous.u*previous.p,
    "contrib_u": e.n*(e.u-previous.u)*previous.p,
    "contrib_p": e.n*e.u*(e.p-previous.p),
    "contrib_c": (e.c-previous.c)*previous.f,
    "contrib_f": e.c*(e.f-previous.f),
    "delta_r": e.r-previous.r, "delta_n": e.n-previous.n,
    "r_prior_year": previous.r, "cancelled_rate_change": e.cancelled_rate-previous.cancelled_rate,
    "yoy_n": e.n/previous.n-1, "yoy_u": e.u/previous.u-1, "yoy_p": e.p/previous.p-1,
}
for metric, expected in formulas.items():
    expected = expected.where(mask)
    close("Phân rã/tăng trưởng "+metric, actual_year[metric],expected,atol=1e-5 if metric.startswith(("contrib","delta","r_")) else 1e-11)
close("Tổng thành phần = delta R",actual_year.loc[mask,["contrib_n","contrib_u","contrib_p"]].sum(axis=1),actual_year.loc[mask,"delta_r"],atol=1e-5)
close("Tổng thành phần = delta N",actual_year.loc[mask,["contrib_c","contrib_f"]].sum(axis=1),actual_year.loc[mask,"delta_n"],atol=1e-8)
segments = frame("SELECT * FROM dbt.reporting.rpt_revenue_segment_yearly")
for dimension in ["category","region","acquisition_channel"]:
    actual = segments[segments.dimension_name.eq(dimension)].set_index(["dimension_value","year"]).sort_index()
    grid = pd.MultiIndex.from_product([sorted(detail[dimension].unique()),e.index],names=["dimension_value","year"])
    expected_cents = detail.groupby([dimension,"year"]).r_cents.sum()
    expected_cents.index.names = grid.names
    expected_cents = expected_cents.reindex(grid,fill_value=0)
    check(dimension+": grid nhóm-năm unique", actual.index.is_unique and actual.index.equals(grid))
    check(dimension+": R exact cents", np.array_equal(cents(actual.r),expected_cents))
    total_cents = pd.Series(actual.index.get_level_values("year").map(e.r_cents),index=actual.index)
    check(dimension+": tổng năm đúng nguồn", np.array_equal(cents(actual.r_total),total_cents))
    year_index = actual.index.get_level_values("year")
    check(dimension+": analysis flag",np.array_equal(actual.is_analysis_period,year_index>=2013))
    share = expected_cents/total_cents
    prior_cents = expected_cents.groupby(level=0).shift()
    delta_cents = expected_cents-prior_cents
    share_prior = share.groupby(level=0).shift()
    total_delta = pd.Series(year_index.map(e.r_cents.diff()),index=actual.index)
    valid = year_index>=2014
    close(dimension+": share",actual.share,share)
    close(dimension+": prior",actual.r_prior_year,(prior_cents/100).where(valid),atol=1e-5)
    close(dimension+": delta",actual.delta_r,(delta_cents/100).where(valid),atol=1e-5)
    close(dimension+": yoy",actual.yoy_rate,(expected_cents/prior_cents.replace(0,np.nan)-1).where(valid))
    close(dimension+": shift pp",actual.share_shift_pp,((share-share_prior)*100).where(valid))
    close(dimension+": contribution",actual.contribution_to_delta,(delta_cents/total_delta.replace(0,np.nan)).where(valid))
display(actual_year.loc[2018:2020,["r","delta_r","contrib_n","contrib_u","contrib_p","delta_n","contrib_c","contrib_f"]])


,r,delta_r,contrib_n,contrib_u,contrib_p,delta_n,contrib_c,contrib_f
year,,,,,,,,
2018,1419275129.27,-42701169.68,-1.292117e+08,-2.381329e+07,1.103238e+08,-5404.0,-2886.048574,-2517.951426
2019,864329801.94,-554945327.33,-5.724206e+08,2.582716e+06,1.489256e+07,-22481.0,-16577.909749,-5903.090251
2020,806414699.14,-57915102.80,-1.373198e+08,-1.249874e+07,9.190342e+07,-5284.0,-3641.278492,-1642.721508


## 7. Phản ví dụ thiết kế: non-additive và fan-out

Hai kiểm tra phải tìm thấy lỗi nếu cố cộng distinct hoặc nối bridge rồi SUM. Ví dụ synthetic drill-across
chỉ minh họa cardinality, không phải dữ liệu kinh doanh mới. Dimension dùng chung không tự chống fan-out.


In [8]:
monthly_customers_sum = detail[detail.delivered].groupby("month_start_date").customer_id.nunique().sum()
all_customers = detail.loc[detail.delivered,"customer_id"].nunique()
category_orders_sum = detail[detail.delivered].groupby("category").order_id.nunique().sum()
all_orders = detail.loc[detail.delivered,"order_id"].nunique()
check("Cộng C tháng làm đếm trùng khách", monthly_customers_sum > all_customers)
check("Cộng N theo category làm đếm trùng đơn", category_orders_sum > all_orders)
print("SUM(C tháng) =",monthly_customers_sum,"| distinct khách toàn kỳ =",all_customers)
print("SUM(N category) =",category_orders_sum,"| distinct đơn toàn kỳ =",all_orders)
fanout = frame("""
SELECT
(SELECT SUM(f.gross_amount) FROM dbt.marts.fact_order_item f
 JOIN dbt.marts.bridge_item_promo b USING(order_item_sk)) AS unsafe_gross,
(SELECT SUM(f.gross_amount) FROM dbt.marts.fact_order_item f
 WHERE EXISTS (SELECT 1 FROM dbt.marts.bridge_item_promo b WHERE b.order_item_sk=f.order_item_sk)) AS safe_gross
""")
check("Bridge M:N làm lặp measure nếu SUM sau JOIN",fanout.unsafe_gross.iloc[0] > fanout.safe_gross.iloc[0])
display(fanout)

# Ví dụ độc lập: hai dòng sales và ba dòng return trên cùng product/day.
sales_demo = pd.DataFrame({"day":["2022-01-01"]*2,"product_id":[1,1],"revenue":[100,200]})
returns_demo = pd.DataFrame({"day":["2022-01-01"]*3,"product_id":[1,1,1],"refund":[10,20,30]})
common_grain = ["day","product_id"]
unsafe = sales_demo.merge(returns_demo,on=common_grain)
safe = sales_demo.groupby(common_grain).revenue.sum().to_frame().join(
    returns_demo.groupby(common_grain).refund.sum(),how="outer").fillna(0)
check("Demo raw fact join gây fan-out",len(unsafe)==6 and unsafe.revenue.sum()==900)
check("Demo aggregate trước giữ đúng số",safe.revenue.sum()==300 and safe.refund.sum()==60)
display(safe)


SUM(C tháng) = 477310 | distinct khách toàn kỳ = 85115
SUM(N category) = 519676 | distinct đơn toàn kỳ = 516716


,unsafe_gross,safe_gross
0,5441739298.25,5435437532.63


,,revenue,refund
day,product_id,,
2022-01-01,1,300,60


## 8. Kết luận và giới hạn nghiệm thu

PASS xác nhận **snapshot local đang được đọc**, không xác nhận trạng thái server PostgreSQL/Snowflake.
Core PS khớp từng dòng với nguồn; reporting được kiểm bằng phép tính riêng; key và FK hiện tại hợp lệ.
Không suy diễn rằng ROW_NUMBER ổn định qua incremental, snapshot chứa SCD2, hoặc sample_submission là forecast.

Chưa triển khai: giai đoạn/CAGR PS2 và bảng điểm đổi hướng; Snowflake production; event pipeline Kafka;
fact_payment_event (chỉ là ví dụ thiết kế); persistent surrogate keys; SCD2 thu thập từ thay đổi thật.
Không sửa model dbt trong đợt thiết kế này. Test tiền so khớp cents; sai số tuyệt đối nhỏ chỉ dùng cho tỷ số/phân rã float.


In [9]:
check("Database read-only không đổi byte",sha256(DB)==manifest.loc[manifest.path.eq(str(DB.relative_to(ROOT))),"sha256"].iloc[0])
for path in source_files:
    check("Nguồn không đổi: "+path.name,sha256(path)==manifest.loc[manifest.path.eq(str(path.relative_to(ROOT))),"sha256"].iloc[0])
results = pd.DataFrame(checks)
display(results)
assert results.passed.all()
print(f"ACCEPTANCE GATE: {len(results)}/{len(results)} PASS")
print("Phạm vi: thiết kế + snapshot local; không rebuild dbt, không deploy production.")
con.close()


,check,passed
0,14 file CSV trong snapshot,True
1,Nguồn join không mất/nhân dòng,True
2,Phản ví dụ order/product không phải PK,True
3,Khóa tái dựng order/line unique,True
4,dim_date: PK không NULL,True
...,...,...
193,Nguồn không đổi: reviews.csv,True
194,Nguồn không đổi: sales.csv,True
195,Nguồn không đổi: sample_submission.csv,True
196,Nguồn không đổi: shipments.csv,True


ACCEPTANCE GATE: 198/198 PASS
Phạm vi: thiết kế + snapshot local; không rebuild dbt, không deploy production.
